# 第2回 宿題（発展）: 用語集で LoRA 学習し、学習前後と「用語集検索」を比べる

Basado en `notebooks/lora_format_tuning.ipynb` (mismas versiones fijadas, mismo modelo `Qwen/Qwen3.5-0.8B`, misma `revision`, mismo LoRA y generación greedy).
Lo que cambia es **el dato** (glosario del curso en lugar del FAQ ficticio) y **la evaluación** (55 preguntas en 7 categorías).

| Condición | Qué cambia | Para qué |
| --- | --- | --- |
| `search` | sin LLM: devuelve la definición encontrada por coincidencia de nombres | "sistema que busca" puro |
| `before` | modelo base, solo la pregunta | punto de partida (conocimiento del preentrenamiento) |
| `rag` | modelo base + extracto del glosario en el system prompt | conocimiento **en la entrada** (Clase 2: ① cambiar la entrada) |
| `after` | modelo + adapter LoRA, solo la pregunta | conocimiento **en los pesos** (Clase 2: ③ cambiar parámetros) |

`rag` se mide **antes** de entrenar, así no hace falta `disable_adapter()` (precaución 1 de la tarea).

Tiempo en T4: ~5 min (0–3) + ~4 min por evaluación × 2 + entrenamiento ~6–12 min (v1, 10 épocas).

## 0. Entorno (igual que el notebook del curso)

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. Configuración

- `VARIANT = "v1"`: datos de `scripts/make_sft_dataset.py` sin cambios (130 ejemplos = 65 términos × 2 plantillas). Es la tarea tal como se pide.
- `VARIANT = "v2"` (opcional, otra sesión): + 2 paráfrasis por término + 20 ejemplos negativos ("no está en el glosario"). Prueba el método ③ de la Clase 2 (enseñar "no sé" con ejemplos, tipo R-Tuning). Tarda ~2× más.
- `num_train_epochs = 10`: con 130 ejemplos son ~170 pasos. El FAQ usó 20 épocas sobre 14 ejemplos (formato); aquí el objetivo es **conocimiento**, que la Clase 2 dice que se aprende lento, pero 20 épocas duplicarían el tiempo. Si cambias este valor, anótalo en `answer.md`.
- El resto es idéntico al notebook del curso para que la comparación sea justa.

In [ ]:
from pathlib import Path
import os, json, time

VARIANT = "v1"   # "v1" (tarea) o "v2" (opcional: + paráfrasis + negativos)

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # mismo peso que el notebook del curso
    "seed": 42,
    "variant": VARIANT,
    "num_train_epochs": 10,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,          # las definiciones llegan a ~210 caracteres
    "gen": {"do_sample": False},    # greedy: mismas condiciones en todas las evaluaciones
    "work_dir": "/content/glossary_lab",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. Subir archivos y verificar los datos

Sube a `/content` (panel izquierdo → icono de carpeta → subir) o ejecuta la celda y elige los archivos:

| Archivo | Cómo se genera (en Codespaces, en tu fork) |
| --- | --- |
| `glossary_sft.jsonl` | `uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl` |
| `glossary_sft_v2.jsonl` (solo v2) | `uv run python scripts/make_glossary_data.py sft-v2 --out glossary_sft_v2.jsonl` |
| `glossary_eval.jsonl` | `uv run python scripts/make_glossary_data.py eval --out glossary_eval.jsonl` |
| `glossary.json` | `data/glossary.json` |
| `glossary_ft.py` | `scripts/glossary_ft.py` |

**Qué mirar**: `datos OK` y los conteos. Si el JSONL no coincide con lo que se regenera desde `glossary.json`, los archivos vienen de versiones distintas del repo: vuelve a generarlos.

In [ ]:
import sys, importlib
TRAIN_FILE = {"v1": "glossary_sft.jsonl", "v2": "glossary_sft_v2.jsonl"}[VARIANT]
NEEDED = [TRAIN_FILE, "glossary_eval.jsonl", "glossary.json", "glossary_ft.py"]
missing = [f for f in NEEDED if not Path("/content", f).exists()]
if missing:
    print("Faltan:", missing, "→ elígelos en el diálogo")
    from google.colab import files
    for name in files.upload():
        print("subido:", name)

sys.path.insert(0, "/content")
import glossary_ft as gft
importlib.reload(gft)

GLOSSARY = gft.load_glossary("/content/glossary.json")
train_records = gft.read_jsonl(f"/content/{TRAIN_FILE}")
eval_items = gft.read_jsonl("/content/glossary_eval.jsonl")

# Verificaciones: integridad, fuga y términos ausentes
assert train_records == gft.build_sft_records(GLOSSARY, VARIANT), "El JSONL no coincide con glossary.json: regenera ambos desde el mismo commit"
assert gft.check_no_leak(train_records, eval_items) == [], "Hay preguntas de evaluación dentro del entrenamiento"
assert gft.check_absent_terms(GLOSSARY) == [], "Un término 'ficticio' o 'ausente' sí está en el glosario"
from collections import Counter
print(f"datos OK | términos={len(GLOSSARY)} | train={len(train_records)} | eval={len(eval_items)}",
      dict(Counter(i["category"] for i in eval_items)))
print("ejemplo:", json.dumps(train_records[0], ensure_ascii=False)[:200])

## 3. Cargar el modelo (igual que el curso) y definir la evaluación

`generate(messages)` recibe la lista de mensajes ya armada, para usar la misma función en `before`, `rag` y `after`.
El formato de entrada de `before`/`after` es **solo el mensaje `user`**, igual que los datos de entrenamiento: así el único cambio entre ambos es el adapter.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(CONFIG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| memoria de pesos (GB):", round(model.get_memory_footprint() / 1e9, 2))

def generate(model, messages):
    # enable_thinking=False igual que el notebook del curso
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

def plain_messages(q):
    return [{"role": "user", "content": q}]

def evaluate(label, answer_fn):
    '''Responde las 55 preguntas, puntúa y guarda eval_<label>.json'''
    t0 = time.time()
    rows = [gft.score_output(it, answer_fn(it["question"]), GLOSSARY) for it in eval_items]
    summary = gft.summarize(rows)
    summary["seconds"] = round(time.time() - t0, 1)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(label, json.dumps(summary["headline"], ensure_ascii=False))
    return summary, rows

RESULTS = {}
RESULTS["search"] = evaluate("search", lambda q: gft.search_answer(q, GLOSSARY))
RESULTS["before"] = evaluate("before", lambda q: generate(model, plain_messages(q)))
RESULTS["rag"] = evaluate("rag", lambda q: generate(model, gft.rag_messages(q, GLOSSARY)))

## 4. Entrenamiento LoRA (misma configuración que el curso)

**Qué mirar**: `trainable%` (unos pocos %), la pérdida cada 10 pasos y el tiempo. Con v1 y 10 épocas son ~170 pasos.
Si se corta la sesión: `trainer.train(resume_from_checkpoint=True)` (checkpoints cada 50 pasos en `ckpt/`).

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)   # ya está en formato messages (igual que to_chat)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4,
                gradient_accumulation_steps=2, learning_rate=CONFIG["lr"], lr_scheduler_type="cosine",
                warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=50, save_total_limit=2,
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
LOSSES = [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h]
print(f"tiempo: {TRAIN_SECONDS} s | pasos: {trainer.state.global_step} | épocas: {trainer.state.epoch:.1f}")
print("pérdida (paso, loss):", LOSSES)
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. Evaluación después del entrenamiento y comparación

In [ ]:
RESULTS["after"] = evaluate("after", lambda q: generate(tuned, plain_messages(q)))

# Métricas principales por condición
headline = pd.DataFrame({k: v[0]["headline"] for k, v in RESULTS.items()})
display(headline)

# Detalle por categoría (veredictos)
by_cat = pd.DataFrame([{"condición": k, "categoría": c, **d} for k, (s, _) in RESULTS.items() for c, d in s["by_category"].items()]).fillna(0)
display(by_cat)

In [ ]:
# Regresiones: bien antes, mal después. "Bien" depende de la categoría.
GOOD = {"seen": {"correct"}, "reworded": {"correct"}, "alias": {"correct"}, "natural": {"correct"},
        "fictional": {"refused"}, "absent_real": {"ok", "refused"}, "general": {"ok"}}
before_rows, after_rows, rag_rows = RESULTS["before"][1], RESULTS["after"][1], RESULTS["rag"][1]
regressed = [b["question"] for b, a in zip(before_rows, after_rows)
             if b["verdict"] in GOOD[b["category"]] and a["verdict"] not in GOOD[a["category"]]]
improved = [b["question"] for b, a in zip(before_rows, after_rows)
            if b["verdict"] not in GOOD[b["category"]] and a["verdict"] in GOOD[a["category"]]]
print("regresiones:", regressed)
print("mejoras:", len(improved))

detail = pd.DataFrame([{"cat": b["category"], "pregunta": b["question"],
                        "before": b["verdict"], "rag": r["verdict"], "after": a["verdict"],
                        "salida after": a["output"][:120], "def. más parecida (after)": f'{a["closest_def"]} ({a["closest_f1"]})'}
                       for b, r, a in zip(before_rows, rag_rows, after_rows)])
display(detail)

### Qué leer en las tablas

- `seen` vs `reworded`/`alias`/`natural`: si `after` acierta en `seen` pero mucho menos en el resto, aprendió **la frase**, no el concepto.
- `acc_unseen_course`: lo que el modelo base no puede saber. Es donde el entrenamiento podría aportar conocimiento nuevo, o fallar (Gekhman et al. 2024).
- `fictional_hallucination`: ¿el modelo entrenado inventa una definición con el estilo del glosario para un término que no existe?
- `contamination_controls`: ¿responde a "日本の首都は？" o a "RLHF" con una definición del glosario? Es olvido o sobreajuste (Clase 2, diapositiva 82).
- Compara `after` con `rag` y `search`: es la discusión central de la tarea.

## 6. Guardar resultados

In [ ]:
import transformers, peft, trl
result = {
    "config": {k: v for k, v in CONFIG.items() if k != "work_dir"},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "data": {"train_file": TRAIN_FILE, "n_train": len(train_records), "n_eval": len(eval_items), "n_terms": len(GLOSSARY)},
    "training": {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "losses": LOSSES},
    "summaries": {k: v[0] for k, v in RESULTS.items()},
    "regressed": regressed,
    "executed_training_myself": True,
}
out = WORK / f"result_homework_{VARIANT}.json"
out.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("guardado:", out)
print(json.dumps({k: result[k] for k in ["data", "training"]}, ensure_ascii=False))

Descarga `result_homework_v1.json` y `eval_*.json` desde `/content/glossary_lab` y luego **Runtime → Disconnect and delete runtime**.

Para v2: **Runtime → Restart session**, cambia `VARIANT = "v2"` y ejecuta desde arriba. La condición `before`/`rag` debe salir igual que en v1, lo que sirve de control.